# 05 — Leiden resolution QC and reproducibility audit

**Purpose:** evaluate the Leiden resolution sweep on the freshly retrained scVI
latent space while preserving the article-defining historical result as a
separate provenance checkpoint.

The archived executed analysis used:

- `X_scVI` with 30 latent dimensions;
- `n_neighbors = 30`;
- Leiden resolutions from `0.10` to `1.20`;
- article-selected resolution `0.60`;
- historical `r=0.60` result: **18 clusters**;
- historical cluster-size summary at `r=0.60`:
  minimum `105`, maximum `61,646`, median `14,336.5` cells.

A fresh scVI retraining is stochastic. Therefore this notebook **does not force**
a fresh rerun to reproduce 18 clusters and does not silently substitute another
resolution. Instead it records whether the fresh rerun reproduces the archived
structural checkpoint.

The article-defining cluster assignments remain a canonical historical artifact
and will be bridged into the downstream article branch separately if the fresh
rerun does not reproduce them exactly.

This notebook is Windows/low-memory safe:

- the scVI output is lightweight (`X` may be absent);
- raw-count integrity is checked from the preserved pre-scVI HVG count H5AD;
- the `n_neighbors=30` graph and completed Leiden resolutions are checkpointed;
- an interrupted resolution sweep resumes from the last completed resolution;
- scratch checkpointing defaults to `E:\glioma-cnv-scratch\main_analysis_05`
  when that drive is available, and can be overridden with
  `GLIOMA_MAIN05_SCRATCH_ROOT`.

## Interpretation rule

The **advisor table is diagnostic for the fresh rerun**. It does not redefine
the article-selected resolution.

- Article/canonical selection: `r = 0.60`, defined by the archived executed run.
- Fresh-rerun result: reported independently.
- If the fresh rerun does not reproduce the archived 18-cluster structure,
  downstream article analysis must use the canonical historical cluster
  assignments rather than relabeling or forcing the rerun.

In [1]:
# =========================
# 0) SETUP
# =========================

import os
import gc
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import scipy.sparse as sp
import h5py

import scanpy as sc
import anndata as ad

import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")


def _find_repository_root(start=None):
    """Locate repository root or use GLIOMA_PROJECT_ROOT when set."""
    start = Path.cwd() if start is None else Path(start)
    start = start.expanduser().resolve()

    for candidate in (start, *start.parents):
        if (
            (candidate / "README.md").exists()
            and (candidate / "code").exists()
        ):
            return candidate

    return start


PROJECT_DIR = Path(
    os.environ.get(
        "GLIOMA_PROJECT_ROOT",
        _find_repository_root(),
    )
).expanduser().resolve()

PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
METADATA_DIR = PROJECT_DIR / "metadata"
FIGURES_DIR = PROJECT_DIR / "figures"
RESULTS_DIR = PROJECT_DIR / "results"

for directory in (
    PROCESSED_DIR,
    METADATA_DIR,
    FIGURES_DIR,
    RESULTS_DIR,
):
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )


SCVI_H5AD = (
    PROCESSED_DIR
    / "integrated"
    / "GBM_core_4datasets_HVG8000_scVI_UMAP_Leiden_lowmem_Xcounts.h5ad"
)

HVG_COUNTS_H5AD = (
    PROCESSED_DIR
    / "integrated"
    / "GBM_core_4datasets_HVG8000_pre_scVI_lowmem_Xcounts.h5ad"
)

FULLGENE_H5AD = (
    PROCESSED_DIR
    / "integrated"
    / "GBM_core_4datasets_fullgene_postQC_merged_counts_metadata.h5ad"
)

# Fresh-rerun output is explicitly named "rerun" so it cannot be confused
# with the article-defining historical 18-cluster artifact.
OUT_H5AD = (
    PROCESSED_DIR
    / "integrated"
    / "GBM_core_4datasets_HVG8000_scVI_clustering_resolution_QC_rerun_lowmem.h5ad"
)


def _default_scratch_root():
    configured = os.environ.get(
        "GLIOMA_MAIN05_SCRATCH_ROOT"
    )

    if configured:
        return Path(configured).expanduser().resolve()

    if os.name == "nt" and Path("E:/").exists():
        return Path(
            r"E:\glioma-cnv-scratch\main_analysis_05"
        )

    return (
        PROJECT_DIR
        / ".scratch"
        / "main_analysis_05"
    ).resolve()


SCRATCH_ROOT = _default_scratch_root()
SCRATCH_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

CHECKPOINT_H5AD = (
    SCRATCH_ROOT
    / "05_leiden_resolution_sweep_checkpoint.h5ad"
)

CHECKPOINT_TMP_H5AD = (
    SCRATCH_ROOT
    / "05_leiden_resolution_sweep_checkpoint.updating.h5ad"
)


# -------------------------
# Fresh-rerun output TSVs
# -------------------------

SWEEP_SUMMARY_TSV = (
    METADATA_DIR
    / "clustering_resolution_sweep_HVG8000_rerun_lowmem.tsv"
)

CLUSTER_SIZES_TSV = (
    METADATA_DIR
    / "clustering_resolution_cluster_sizes_HVG8000_rerun_lowmem.tsv"
)

COMPOSITION_TSV = (
    METADATA_DIR
    / "clustering_resolution_composition_metrics_HVG8000_rerun_lowmem.tsv"
)

COMPOSITION_SUMMARY_TSV = (
    METADATA_DIR
    / "clustering_resolution_composition_summary_HVG8000_rerun_lowmem.tsv"
)

SILHOUETTE_TSV = (
    METADATA_DIR
    / "clustering_resolution_silhouette_metrics_HVG8000_rerun_lowmem.tsv"
)

MARKER_MODULE_TSV = (
    METADATA_DIR
    / "clustering_resolution_marker_module_summary_HVG8000_rerun_lowmem.tsv"
)

ADVISOR_TSV = (
    METADATA_DIR
    / "clustering_resolution_advisor_table_HVG8000_rerun_lowmem.tsv"
)

RECOMMENDATION_TSV = (
    METADATA_DIR
    / "clustering_resolution_recommended_resolution_HVG8000_rerun_lowmem.tsv"
)

STATUS_TSV = (
    METADATA_DIR
    / "clustering_resolution_reproducibility_status_HVG8000.tsv"
)

MANIFEST_TSV = (
    METADATA_DIR
    / "clustering_resolution_QC_output_manifest_HVG8000_rerun_lowmem.tsv"
)


# -------------------------
# Parameters
# -------------------------

EMBED_KEY = "X_scVI"
UMAP_KEY = "X_umap"

N_NEIGHBORS = 30
NEIGHBORS_KEY = f"neighbors_scVI_n{N_NEIGHBORS}"

RESOLUTIONS = [
    0.10, 0.15, 0.20, 0.25,
    0.30, 0.35, 0.40, 0.45,
    0.50, 0.55, 0.60, 0.70,
    0.80, 0.90, 1.00, 1.20,
]

RANDOM_STATE = 42

EXPECTED_SHAPE = (
    351_427,
    8_000,
)

EXPECTED_LATENT_SHAPE = (
    351_427,
    30,
)

# Article-defining historical checkpoint from the archived executed run.
ARTICLE_SELECTED_RESOLUTION = 0.60
HISTORICAL_R06_N_CLUSTERS = 18
HISTORICAL_R06_MIN_CLUSTER_SIZE = 105
HISTORICAL_R06_MAX_CLUSTER_SIZE = 61_646
HISTORICAL_R06_MEDIAN_CLUSTER_SIZE = 14_336.5

SILHOUETTE_MAX_CELLS = 30_000
N_TOP_CANDIDATE_FIGURES = 4

sc.settings.verbosity = 2
sc.settings.figdir = str(
    FIGURES_DIR
)


def write_tsv_replace(
    df,
    path,
):
    path = Path(path)
    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    if path.exists():
        path.unlink()

    df.to_csv(
        path,
        sep="\t",
        index=False,
    )

    print(
        f"[SAVED] {path} "
        f"shape={df.shape}"
    )


def write_h5ad_replace(
    adata_obj,
    path,
):
    path = Path(path)
    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    tmp = path.with_suffix(
        path.suffix + ".tmp"
    )

    if tmp.exists():
        tmp.unlink()

    adata_obj.write_h5ad(
        tmp,
        compression="lzf",
    )

    os.replace(
        tmp,
        path,
    )

    print(
        f"[SAVED] {path}"
    )


def write_checkpoint(
    adata_obj,
):
    if CHECKPOINT_TMP_H5AD.exists():
        CHECKPOINT_TMP_H5AD.unlink()

    adata_obj.write_h5ad(
        CHECKPOINT_TMP_H5AD,
        compression="lzf",
    )

    os.replace(
        CHECKPOINT_TMP_H5AD,
        CHECKPOINT_H5AD,
    )

    print(
        "[CHECKPOINT SAVED]",
        CHECKPOINT_H5AD,
    )


def normalized_entropy(values):
    series = pd.Series(
        values
    ).astype(str)

    counts = series.value_counts()

    if counts.shape[0] <= 1:
        return 0.0

    p = counts / counts.sum()

    entropy = -np.sum(
        p * np.log(
            p + 1e-12
        )
    )

    return float(
        entropy
        / np.log(
            counts.shape[0]
        )
    )


def safe_fraction_top(values):
    series = pd.Series(
        values
    ).astype(str)

    if len(series) == 0:
        return (
            np.nan,
            "NA",
        )

    counts = series.value_counts()

    return (
        float(
            counts.iloc[0]
            / counts.sum()
        ),
        str(
            counts.index[0]
        ),
    )


def leiden_key_for_resolution(
    resolution,
):
    resolution_text = str(
        resolution
    ).replace(
        ".",
        "_",
    )

    return (
        "leiden_scvi_HVG8000_r"
        + resolution_text
    )


print(
    "PROJECT_DIR:",
    PROJECT_DIR,
)

print(
    "SCVI_H5AD:",
    SCVI_H5AD,
    "exists:",
    SCVI_H5AD.exists(),
)

print(
    "HVG_COUNTS_H5AD:",
    HVG_COUNTS_H5AD,
    "exists:",
    HVG_COUNTS_H5AD.exists(),
)

print(
    "FULLGENE_H5AD:",
    FULLGENE_H5AD,
    "exists:",
    FULLGENE_H5AD.exists(),
)

print(
    "SCRATCH_ROOT:",
    SCRATCH_ROOT,
)

print(
    "CHECKPOINT_H5AD:",
    CHECKPOINT_H5AD,
    "exists:",
    CHECKPOINT_H5AD.exists(),
)

PROJECT_DIR: G:\Repository\glioma-cnv-single-cell-atlas
SCVI_H5AD: G:\Repository\glioma-cnv-single-cell-atlas\data\processed\integrated\GBM_core_4datasets_HVG8000_scVI_UMAP_Leiden_lowmem_Xcounts.h5ad exists: True
HVG_COUNTS_H5AD: G:\Repository\glioma-cnv-single-cell-atlas\data\processed\integrated\GBM_core_4datasets_HVG8000_pre_scVI_lowmem_Xcounts.h5ad exists: True
FULLGENE_H5AD: G:\Repository\glioma-cnv-single-cell-atlas\data\processed\integrated\GBM_core_4datasets_fullgene_postQC_merged_counts_metadata.h5ad exists: True
SCRATCH_ROOT: E:\glioma-cnv-scratch\main_analysis_05
CHECKPOINT_H5AD: E:\glioma-cnv-scratch\main_analysis_05\05_leiden_resolution_sweep_checkpoint.h5ad exists: True


In [2]:
# =========================
# 1) READ LIGHTWEIGHT scVI OBJECT AND VALIDATE INPUTS
# =========================

if not SCVI_H5AD.exists():
    raise FileNotFoundError(
        f"scVI H5AD not found: {SCVI_H5AD}"
    )

if not HVG_COUNTS_H5AD.exists():
    raise FileNotFoundError(
        "Preserved pre-scVI HVG count H5AD is required: "
        f"{HVG_COUNTS_H5AD}"
    )


source_adata = sc.read_h5ad(
    SCVI_H5AD
)

print(
    source_adata
)

if tuple(
    source_adata.shape
) != EXPECTED_SHAPE:
    raise AssertionError(
        "Unexpected scVI object shape: "
        f"{source_adata.shape}"
    )


required_obs = [
    "core_dataset",
    "condition",
    "patient_id",
    "sample_id",
    "scvi_batch",
]

missing_obs = [
    column
    for column in required_obs
    if column
    not in source_adata.obs.columns
]

if missing_obs:
    raise KeyError(
        "Missing required obs columns: "
        f"{missing_obs}"
    )


if EMBED_KEY not in source_adata.obsm:
    raise KeyError(
        f"{EMBED_KEY} missing from obsm."
    )

if tuple(
    source_adata.obsm[
        EMBED_KEY
    ].shape
) != EXPECTED_LATENT_SHAPE:
    raise AssertionError(
        f"Unexpected {EMBED_KEY} shape: "
        f"{source_adata.obsm[EMBED_KEY].shape}"
    )


latent_nan_count = int(
    np.isnan(
        source_adata.obsm[
            EMBED_KEY
        ]
    ).sum()
)

if latent_nan_count:
    raise ValueError(
        f"{EMBED_KEY} contains "
        f"{latent_nan_count} NaN values."
    )


for column in required_obs:
    source_adata.obs[column] = (
        source_adata.obs[column]
        .astype(str)
        .replace(
            {
                "nan": "unknown",
                "None": "unknown",
                "<NA>": "unknown",
                "": "unknown",
            }
        )
    )


def _decode_attr(value):
    if isinstance(
        value,
        bytes,
    ):
        return value.decode(
            "utf-8"
        )

    return value


def _stored_shape(node):
    if isinstance(
        node,
        h5py.Dataset,
    ):
        return tuple(
            int(x)
            for x in node.shape
        )

    shape = node.attrs.get(
        "shape"
    )

    if shape is None:
        raise KeyError(
            f"{node.name} has no stored shape."
        )

    return tuple(
        int(x)
        for x in shape
    )


with h5py.File(
    HVG_COUNTS_H5AD,
    "r",
) as handle:

    if "X" not in handle:
        raise KeyError(
            "Pre-scVI HVG H5AD is missing /X."
        )

    x_node = handle["X"]
    counts_shape = _stored_shape(
        x_node
    )

    if counts_shape != EXPECTED_SHAPE:
        raise AssertionError(
            "Unexpected raw-count shape: "
            f"{counts_shape}"
        )

    if not isinstance(
        x_node,
        h5py.Group,
    ):
        raise AssertionError(
            "Preserved counts are not sparse CSR."
        )

    x_encoding = str(
        _decode_attr(
            x_node.attrs.get(
                "encoding-type",
                "",
            )
        )
    )

    if x_encoding != "csr_matrix":
        raise AssertionError(
            "Expected csr_matrix, observed "
            f"{x_encoding!r}."
        )

    data = x_node["data"]

    n_values = int(
        data.shape[0]
    )

    sample_n = min(
        n_values,
        1_000_000,
    )

    values = np.asarray(
        data[:sample_n]
    )

    if values.size == 0:
        raise ValueError(
            "Raw-count matrix has no nonzero values."
        )

    if not np.isfinite(
        values
    ).all():
        raise ValueError(
            "Raw counts contain non-finite values."
        )

    x_sample_min = float(
        values.min()
    )

    x_sample_max = float(
        values.max()
    )

    x_sample_any_negative = bool(
        np.any(
            values < 0
        )
    )

    if np.issubdtype(
        values.dtype,
        np.integer,
    ):
        x_sample_integer_like = True

    else:
        x_sample_integer_like = bool(
            np.allclose(
                values,
                np.round(
                    values
                ),
            )
        )

    counts_dtype = str(
        data.dtype
    )

    counts_nnz = n_values


if x_sample_any_negative:
    raise ValueError(
        "Raw counts contain negative values."
    )

if not x_sample_integer_like:
    raise ValueError(
        "Pre-scVI X does not look like integer raw counts."
    )


scvi_x_storage = (
    "omitted_by_design"
    if source_adata.X is None
    else (
        "sparse"
        if sp.issparse(
            source_adata.X
        )
        else "dense"
    )
)


INPUT_QC_TSV = (
    METADATA_DIR
    / "clustering_resolution_input_qc_HVG8000_rerun_lowmem.tsv"
)


input_qc = pd.DataFrame(
    [
        {
            "n_cells": source_adata.n_obs,
            "n_genes": source_adata.n_vars,
            "scvi_output_X_storage": scvi_x_storage,
            "scvi_output_layers": ",".join(
                list(source_adata.layers.keys())
            ),
            "counts_qc_source": str(HVG_COUNTS_H5AD),
            "counts_encoding": x_encoding,
            "counts_dtype": counts_dtype,
            "counts_nnz": counts_nnz,
            "X_sample_min": x_sample_min,
            "X_sample_max": x_sample_max,
            "X_sample_integer_like": x_sample_integer_like,
            "X_sample_any_negative": x_sample_any_negative,
            "latent_key": EMBED_KEY,
            "latent_shape": str(source_adata.obsm[EMBED_KEY].shape),
            "latent_nan_count": latent_nan_count,
        }
    ]
)


write_tsv_replace(
    input_qc,
    INPUT_QC_TSV,
)

display(
    input_qc
)

print(
    "Input validated."
)

print(
    "scVI expression storage:",
    scvi_x_storage,
)

print(
    "Raw-count QC source:",
    HVG_COUNTS_H5AD,
)

AnnData object with n_obs × n_vars = 351427 × 8000
    obs: 'sample_id', 'geo_id', 'core_dataset', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'total_counts_ribo', 'pct_counts_ribo', 'total_counts_hb', 'pct_counts_hb', 'dataset_id', 'study_id', 'source_dataset', 'source_file', 'concat_key', 'patient_id', 'condition', 'batch_dataset', 'batch_sample', 'batch_patient', 'original_cell_barcode', 'diagnosis', 'disease_stage', 'tumor_status', 'platform', 'technology', 'study_batch', 'sample_batch', 'scvi_batch', 'analysis_group_major', 'analysis_group_detail', 'ready_for_integration', 'cell_id_original', 'leiden_scvi_0.1', 'leiden_scvi_0.2', 'leiden_scvi_0.3', 'leiden_scvi_0.4', 'leiden_scvi_0.5'
    var: 'highly_variable', 'highly_variable_rank', 'means', 'variances', 'variances_norm', 'highly_variable_nbatches'
    uns: 'leiden_scvi_0.1', 'leiden_scvi_0.2', 'leiden_scvi_0.3', 'leiden_scvi_0.4', 'leiden_scvi_0.5', 'neighbors_scVI', 'scVI_run_info', 'scVI_umap_lei

,n_cells,n_genes,scvi_output_X_storage,scvi_output_layers,counts_qc_source,counts_encoding,counts_dtype,counts_nnz,X_sample_min,X_sample_max,X_sample_integer_like,X_sample_any_negative,latent_key,latent_shape,latent_nan_count
0,351427,8000,omitted_by_design,,G:\Repository\glioma-cnv-single-cell-atlas\dat...,csr_matrix,int32,316259948,1.0,756.0,True,False,X_scVI,"(351427, 30)",0


Input validated.
scVI expression storage: omitted_by_design
Raw-count QC source: G:\Repository\glioma-cnv-single-cell-atlas\data\processed\integrated\GBM_core_4datasets_HVG8000_pre_scVI_lowmem_Xcounts.h5ad


In [3]:
# =========================
# 2) BUILD/RESUME n=30 GRAPH + LEIDEN RESOLUTION SWEEP
# =========================

def _checkpoint_matches_source(
    checkpoint,
    source,
):
    if tuple(checkpoint.shape) != EXPECTED_SHAPE:
        return False

    if EMBED_KEY not in checkpoint.obsm:
        return False

    if tuple(
        checkpoint.obsm[EMBED_KEY].shape
    ) != EXPECTED_LATENT_SHAPE:
        return False

    if not checkpoint.obs_names.equals(
        source.obs_names
    ):
        return False

    # Exact latent comparison prevents accidentally resuming a checkpoint
    # made from a different scVI rerun.
    if not np.array_equal(
        np.asarray(
            checkpoint.obsm[EMBED_KEY]
        ),
        np.asarray(
            source.obsm[EMBED_KEY]
        ),
    ):
        return False

    return True


checkpoint_reused = False

if CHECKPOINT_H5AD.exists():
    print(
        "Loading existing checkpoint:",
        CHECKPOINT_H5AD,
    )

    candidate = sc.read_h5ad(
        CHECKPOINT_H5AD
    )

    if _checkpoint_matches_source(
        candidate,
        source_adata,
    ):
        adata = candidate
        checkpoint_reused = True

        print(
            "Checkpoint matches the current scVI source."
        )

    else:
        print(
            "[STALE CHECKPOINT] Source/latent mismatch; "
            "starting from the current scVI source."
        )

        del candidate
        adata = source_adata.copy()

else:
    adata = source_adata.copy()


del source_adata
gc.collect()


# Remove notebook-04 n_neighbors=15 graph only if present.
for key in (
    "neighbors_scVI_connectivities",
    "neighbors_scVI_distances",
):
    if key in adata.obsp:
        del adata.obsp[key]

if "neighbors_scVI" in adata.uns:
    del adata.uns["neighbors_scVI"]


required_graph_keys = [
    f"{NEIGHBORS_KEY}_connectivities",
    f"{NEIGHBORS_KEY}_distances",
]

graph_ready = (
    NEIGHBORS_KEY in adata.uns
    and all(
        key in adata.obsp
        for key in required_graph_keys
    )
)


if graph_ready:
    print(
        "Historical-analysis graph already present:",
        f"{NEIGHBORS_KEY} "
        f"(n_neighbors={N_NEIGHBORS}).",
    )

else:
    print(
        f"Building neighbors using {EMBED_KEY}, "
        f"n_neighbors={N_NEIGHBORS}"
    )

    sc.pp.neighbors(
        adata,
        use_rep=EMBED_KEY,
        n_neighbors=N_NEIGHBORS,
        key_added=NEIGHBORS_KEY,
        random_state=RANDOM_STATE,
    )

    write_checkpoint(
        adata
    )


completed_before = []
missing_before = []

for resolution in RESOLUTIONS:
    key = leiden_key_for_resolution(
        resolution
    )

    if key in adata.obs.columns:
        completed_before.append(
            resolution
        )
    else:
        missing_before.append(
            resolution
        )


print(
    "\nAlready completed:",
    completed_before,
)

print(
    "Still to run:",
    missing_before,
)


for resolution in RESOLUTIONS:
    key = leiden_key_for_resolution(
        resolution
    )

    if key in adata.obs.columns:
        print(
            "[SKIP already completed]",
            f"resolution={resolution}",
        )
        continue

    print(
        "\n[RUN]",
        f"Leiden resolution={resolution}",
        "->",
        key,
    )

    sc.tl.leiden(
        adata,
        resolution=resolution,
        key_added=key,
        neighbors_key=NEIGHBORS_KEY,
        random_state=RANDOM_STATE,
    )

    counts = (
        adata.obs[key]
        .astype(str)
        .value_counts()
    )

    print(
        "[COMPLETED]",
        f"resolution={resolution}",
        f"n_clusters={counts.shape[0]}",
        f"min={int(counts.min())}",
        f"max={int(counts.max())}",
    )

    write_checkpoint(
        adata
    )

    gc.collect()


# Reconstruct complete tables from all columns, including resumed columns.
sweep_rows = []
cluster_size_rows = []

for resolution in RESOLUTIONS:
    key = leiden_key_for_resolution(
        resolution
    )

    if key not in adata.obs.columns:
        raise AssertionError(
            "Resolution sweep incomplete: "
            f"{key} missing."
        )

    counts = (
        adata.obs[key]
        .astype(str)
        .value_counts()
        .sort_index(
            key=lambda index: [
                int(x)
                if str(x).isdigit()
                else str(x)
                for x in index
            ]
        )
    )

    for cluster, n_cells in counts.items():
        cluster_size_rows.append(
            {
                "resolution": resolution,
                "cluster_key": key,
                "cluster": str(cluster),
                "n_cells": int(n_cells),
                "fraction_cells": float(
                    n_cells / adata.n_obs
                ),
            }
        )

    sweep_rows.append(
        {
            "resolution": resolution,
            "cluster_key": key,
            "n_clusters": int(counts.shape[0]),
            "min_cluster_size": int(counts.min()),
            "max_cluster_size": int(counts.max()),
            "median_cluster_size": float(counts.median()),
            "mean_cluster_size": float(counts.mean()),
            "n_clusters_lt_50_cells": int((counts < 50).sum()),
            "n_clusters_lt_100_cells": int((counts < 100).sum()),
            "n_clusters_lt_500_cells": int((counts < 500).sum()),
            "n_clusters_lt_1000_cells": int((counts < 1000).sum()),
            "frac_cells_in_clusters_lt_100": float(
                counts[counts < 100].sum() / adata.n_obs
            ),
            "frac_cells_in_clusters_lt_500": float(
                counts[counts < 500].sum() / adata.n_obs
            ),
            "frac_cells_in_clusters_lt_1000": float(
                counts[counts < 1000].sum() / adata.n_obs
            ),
        }
    )


sweep_df = pd.DataFrame(
    sweep_rows
)

cluster_sizes_df = pd.DataFrame(
    cluster_size_rows
)


write_tsv_replace(
    sweep_df,
    SWEEP_SUMMARY_TSV,
)

write_tsv_replace(
    cluster_sizes_df,
    CLUSTER_SIZES_TSV,
)


resolution_keys = list(
    sweep_df["cluster_key"]
)


print(
    "\nFULL SWEEP COMPLETED"
)

print(
    "Completed resolutions:",
    len(RESOLUTIONS),
)

print(
    "Checkpoint reused:",
    checkpoint_reused,
)

display(
    sweep_df
)

Loading existing checkpoint: E:\glioma-cnv-scratch\main_analysis_05\05_leiden_resolution_sweep_checkpoint.h5ad
Checkpoint matches the current scVI source.
Historical-analysis graph already present: neighbors_scVI_n30 (n_neighbors=30).

Already completed: [0.1, 0.15, 0.2, 0.25, 0.3, 0.35, 0.4, 0.45, 0.5, 0.55, 0.6, 0.7, 0.8, 0.9, 1.0, 1.2]
Still to run: []
[SKIP already completed] resolution=0.1
[SKIP already completed] resolution=0.15
[SKIP already completed] resolution=0.2
[SKIP already completed] resolution=0.25
[SKIP already completed] resolution=0.3
[SKIP already completed] resolution=0.35
[SKIP already completed] resolution=0.4
[SKIP already completed] resolution=0.45
[SKIP already completed] resolution=0.5
[SKIP already completed] resolution=0.55
[SKIP already completed] resolution=0.6
[SKIP already completed] resolution=0.7
[SKIP already completed] resolution=0.8
[SKIP already completed] resolution=0.9
[SKIP already completed] resolution=1.0
[SKIP already completed] resolution=1

,resolution,cluster_key,n_clusters,min_cluster_size,max_cluster_size,median_cluster_size,mean_cluster_size,n_clusters_lt_50_cells,n_clusters_lt_100_cells,n_clusters_lt_500_cells,n_clusters_lt_1000_cells,frac_cells_in_clusters_lt_100,frac_cells_in_clusters_lt_500,frac_cells_in_clusters_lt_1000
0,0.10,leiden_scvi_HVG8000_r0_1,6,1145,167824,18198.0,58571.166667,0,0,0,0,0.00000,0.000000,0.000000
1,0.15,leiden_scvi_HVG8000_r0_15,6,1146,169896,18211.0,58571.166667,0,0,0,0,0.00000,0.000000,0.000000
2,0.20,leiden_scvi_HVG8000_r0_2,7,1147,138023,21647.0,50203.857143,0,0,0,0,0.00000,0.000000,0.000000
3,0.25,leiden_scvi_HVG8000_r0_25,9,1179,129955,14913.0,39047.444444,0,0,0,0,0.00000,0.000000,0.000000
4,0.30,leiden_scvi_HVG8000_r0_3,8,1187,137909,24679.0,43928.375000,0,0,0,0,0.00000,0.000000,0.000000
5,0.35,leiden_scvi_HVG8000_r0_35,11,1185,99826,21812.0,31947.909091,0,0,0,0,0.00000,0.000000,0.000000
6,0.40,leiden_scvi_HVG8000_r0_4,9,2653,131539,21762.0,39047.444444,0,0,0,0,0.00000,0.000000,0.000000
7,0.45,leiden_scvi_HVG8000_r0_45,15,189,97771,14933.0,23428.466667,0,0,1,2,0.00000,0.000538,0.002200
8,0.50,leiden_scvi_HVG8000_r0_5,12,1278,100715,25397.0,29285.583333,0,0,0,0,0.00000,0.000000,0.000000
9,0.55,leiden_scvi_HVG8000_r0_55,16,1380,72364,12440.5,21964.187500,0,0,0,0,0.00000,0.000000,0.000000


In [4]:
# =========================
# 3) FRESH-RERUN STRUCTURAL STATUS vs ARCHIVED ARTICLE CHECKPOINT
# =========================

r06_row = sweep_df.loc[
    np.isclose(
        sweep_df["resolution"],
        ARTICLE_SELECTED_RESOLUTION,
    )
].iloc[0]


fresh_r06_n_clusters = int(
    r06_row["n_clusters"]
)

fresh_r06_min = int(
    r06_row["min_cluster_size"]
)

fresh_r06_max = int(
    r06_row["max_cluster_size"]
)

fresh_r06_median = float(
    r06_row["median_cluster_size"]
)


fresh_rerun_reproduces_archived_cluster_count = (
    fresh_r06_n_clusters
    == HISTORICAL_R06_N_CLUSTERS
)

fresh_rerun_reproduces_archived_size_summary = bool(
    fresh_r06_n_clusters
    == HISTORICAL_R06_N_CLUSTERS
    and fresh_r06_min
    == HISTORICAL_R06_MIN_CLUSTER_SIZE
    and fresh_r06_max
    == HISTORICAL_R06_MAX_CLUSTER_SIZE
    and np.isclose(
        fresh_r06_median,
        HISTORICAL_R06_MEDIAN_CLUSTER_SIZE,
    )
)


status_df = pd.DataFrame(
    [
        {
            "article_selected_resolution": ARTICLE_SELECTED_RESOLUTION,
            "historical_n_clusters": HISTORICAL_R06_N_CLUSTERS,
            "fresh_rerun_n_clusters": fresh_r06_n_clusters,
            "historical_min_cluster_size": HISTORICAL_R06_MIN_CLUSTER_SIZE,
            "fresh_rerun_min_cluster_size": fresh_r06_min,
            "historical_max_cluster_size": HISTORICAL_R06_MAX_CLUSTER_SIZE,
            "fresh_rerun_max_cluster_size": fresh_r06_max,
            "historical_median_cluster_size": HISTORICAL_R06_MEDIAN_CLUSTER_SIZE,
            "fresh_rerun_median_cluster_size": fresh_r06_median,
            "fresh_rerun_reproduces_archived_cluster_count":
                fresh_rerun_reproduces_archived_cluster_count,
            "fresh_rerun_reproduces_archived_size_summary":
                fresh_rerun_reproduces_archived_size_summary,
            "article_downstream_source": (
                "fresh_rerun"
                if fresh_rerun_reproduces_archived_size_summary
                else "canonical_historical_artifact_required"
            ),
        }
    ]
)


write_tsv_replace(
    status_df,
    STATUS_TSV,
)

display(
    status_df
)


if fresh_rerun_reproduces_archived_size_summary:
    print(
        "Fresh rerun reproduced the archived r=0.60 "
        "structural checkpoint."
    )

else:
    print(
        "\n[REPRODUCIBILITY AUDIT WARNING]"
    )

    print(
        "Fresh scVI/Leiden rerun did NOT exactly reproduce "
        "the archived article-defining r=0.60 partition."
    )

    print(
        "Do not force another resolution and do not replace "
        "the article's historical 18-cluster assignments with this rerun."
    )

    print(
        "This notebook will continue to generate fresh-rerun QC outputs only."
    )


display(
    sweep_df[
        [
            "resolution",
            "n_clusters",
            "min_cluster_size",
            "max_cluster_size",
            "median_cluster_size",
            "n_clusters_lt_100_cells",
            "n_clusters_lt_500_cells",
            "n_clusters_lt_1000_cells",
        ]
    ]
)

[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\clustering_resolution_reproducibility_status_HVG8000.tsv shape=(1, 12)


,article_selected_resolution,historical_n_clusters,fresh_rerun_n_clusters,historical_min_cluster_size,fresh_rerun_min_cluster_size,historical_max_cluster_size,fresh_rerun_max_cluster_size,historical_median_cluster_size,fresh_rerun_median_cluster_size,fresh_rerun_reproduces_archived_cluster_count,fresh_rerun_reproduces_archived_size_summary,article_downstream_source
0,0.6,18,15,105,586,61646,71112,14336.5,21853.0,False,False,canonical_historical_artifact_required



[REPRODUCIBILITY AUDIT WARNING]
Fresh scVI/Leiden rerun did NOT exactly reproduce the archived article-defining r=0.60 partition.
Do not force another resolution and do not replace the article's historical 18-cluster assignments with this rerun.
This notebook will continue to generate fresh-rerun QC outputs only.


,resolution,n_clusters,min_cluster_size,max_cluster_size,median_cluster_size,n_clusters_lt_100_cells,n_clusters_lt_500_cells,n_clusters_lt_1000_cells
0,0.10,6,1145,167824,18198.0,0,0,0
1,0.15,6,1146,169896,18211.0,0,0,0
2,0.20,7,1147,138023,21647.0,0,0,0
3,0.25,9,1179,129955,14913.0,0,0,0
4,0.30,8,1187,137909,24679.0,0,0,0
5,0.35,11,1185,99826,21812.0,0,0,0
6,0.40,9,2653,131539,21762.0,0,0,0
7,0.45,15,189,97771,14933.0,0,1,2
8,0.50,12,1278,100715,25397.0,0,0,0
9,0.55,16,1380,72364,12440.5,0,0,0


In [5]:
# =========================
# 3) CLUSTER COMPOSITION / BIAS METRICS
# =========================

composition_rows = []

metadata_cols = {
    "core_dataset": "dataset",
    "condition": "condition",
    "patient_id": "patient",
    "sample_id": "sample",
}

for _, row in sweep_df.iterrows():
    res = float(row["resolution"])
    key = row["cluster_key"]

    for cl, sub_idx in adata.obs.groupby(key, observed=True).groups.items():
        obs_sub = adata.obs.loc[sub_idx]
        out = {
            "resolution": res,
            "cluster_key": key,
            "cluster": str(cl),
            "n_cells": int(obs_sub.shape[0]),
            "fraction_cells": float(obs_sub.shape[0] / adata.n_obs),
        }

        for col, label in metadata_cols.items():
            top_frac, top_label = safe_fraction_top(obs_sub[col])
            out[f"top_{label}"] = top_label
            out[f"top_{label}_fraction"] = top_frac
            out[f"{label}_normalized_entropy"] = normalized_entropy(obs_sub[col])

        composition_rows.append(out)

composition_df = pd.DataFrame(composition_rows)
write_tsv_replace(composition_df, COMPOSITION_TSV)

summary_rows = []
for (res, key), sub in composition_df.groupby(["resolution", "cluster_key"], observed=True):
    weights = sub["n_cells"].values / sub["n_cells"].sum()

    def weighted_mean(col):
        return float(np.sum(sub[col].values * weights))

    summary_rows.append({
        "resolution": res,
        "cluster_key": key,
        "n_clusters": int(sub.shape[0]),
        "mean_top_dataset_fraction_weighted": weighted_mean("top_dataset_fraction"),
        "mean_top_condition_fraction_weighted": weighted_mean("top_condition_fraction"),
        "mean_top_patient_fraction_weighted": weighted_mean("top_patient_fraction"),
        "mean_top_sample_fraction_weighted": weighted_mean("top_sample_fraction"),
        "mean_dataset_entropy_weighted": weighted_mean("dataset_normalized_entropy"),
        "mean_condition_entropy_weighted": weighted_mean("condition_normalized_entropy"),
        "mean_patient_entropy_weighted": weighted_mean("patient_normalized_entropy"),
        "mean_sample_entropy_weighted": weighted_mean("sample_normalized_entropy"),
        "n_clusters_dataset_fraction_gt_0_80": int((sub["top_dataset_fraction"] > 0.80).sum()),
        "n_clusters_condition_fraction_gt_0_80": int((sub["top_condition_fraction"] > 0.80).sum()),
        "n_clusters_patient_fraction_gt_0_80": int((sub["top_patient_fraction"] > 0.80).sum()),
        "n_clusters_sample_fraction_gt_0_80": int((sub["top_sample_fraction"] > 0.80).sum()),
        "max_top_dataset_fraction": float(sub["top_dataset_fraction"].max()),
        "max_top_condition_fraction": float(sub["top_condition_fraction"].max()),
        "max_top_patient_fraction": float(sub["top_patient_fraction"].max()),
        "max_top_sample_fraction": float(sub["top_sample_fraction"].max()),
    })

composition_summary_df = pd.DataFrame(summary_rows)
write_tsv_replace(composition_summary_df, COMPOSITION_SUMMARY_TSV)

display(composition_summary_df)

[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\clustering_resolution_composition_metrics_HVG8000_rerun_lowmem.tsv shape=(228, 17)
[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\clustering_resolution_composition_summary_HVG8000_rerun_lowmem.tsv shape=(16, 19)


,resolution,cluster_key,n_clusters,mean_top_dataset_fraction_weighted,mean_top_condition_fraction_weighted,mean_top_patient_fraction_weighted,mean_top_sample_fraction_weighted,mean_dataset_entropy_weighted,mean_condition_entropy_weighted,mean_patient_entropy_weighted,mean_sample_entropy_weighted,n_clusters_dataset_fraction_gt_0_80,n_clusters_condition_fraction_gt_0_80,n_clusters_patient_fraction_gt_0_80,n_clusters_sample_fraction_gt_0_80,max_top_dataset_fraction,max_top_condition_fraction,max_top_patient_fraction,max_top_sample_fraction
0,0.10,leiden_scvi_HVG8000_r0_1,6,0.629456,0.457321,0.210533,0.074229,0.656105,0.823598,0.756862,0.870168,2,0,0,0,0.991266,0.617933,0.580441,0.373799
1,0.15,leiden_scvi_HVG8000_r0_15,6,0.629343,0.455198,0.210601,0.074032,0.657060,0.824878,0.756995,0.870467,2,0,0,0,0.991274,0.617687,0.580290,0.373473
2,0.20,leiden_scvi_HVG8000_r0_2,7,0.629405,0.455309,0.212263,0.077880,0.655404,0.820482,0.753066,0.868785,2,0,0,0,0.991282,0.616898,0.579988,0.374019
3,0.25,leiden_scvi_HVG8000_r0_25,9,0.629428,0.471051,0.228955,0.090548,0.650546,0.810675,0.746194,0.858710,4,0,0,0,0.991518,0.761703,0.749502,0.502988
4,0.30,leiden_scvi_HVG8000_r0_3,8,0.629439,0.455324,0.220379,0.079820,0.650380,0.817386,0.745203,0.859403,2,0,0,0,0.990733,0.616480,0.579382,0.364785
5,0.35,leiden_scvi_HVG8000_r0_35,11,0.629869,0.507747,0.233340,0.117088,0.645246,0.783339,0.722213,0.829947,3,0,0,0,0.990717,0.783689,0.763958,0.545016
6,0.40,leiden_scvi_HVG8000_r0_4,9,0.629454,0.470883,0.236647,0.091675,0.647579,0.808339,0.737342,0.848915,3,0,0,0,0.969166,0.710728,0.685101,0.437124
7,0.45,leiden_scvi_HVG8000_r0_45,15,0.630524,0.511842,0.240428,0.119743,0.640032,0.778407,0.715109,0.821521,6,0,0,0,0.988014,0.723718,0.695026,0.629630
8,0.50,leiden_scvi_HVG8000_r0_5,12,0.629784,0.511065,0.237073,0.117717,0.643319,0.781488,0.717280,0.824780,3,0,0,0,0.989828,0.781487,0.757988,0.542914
9,0.55,leiden_scvi_HVG8000_r0_55,16,0.633369,0.521844,0.242656,0.128525,0.635295,0.768134,0.708505,0.812696,6,0,0,0,0.988406,0.783859,0.769773,0.547029


In [6]:
# =========================
# 4) SILHOUETTE METRICS ON SUBSAMPLED scVI LATENT
# =========================

# This is an approximate diagnostic, not the final decision by itself.
# cluster silhouette: higher can mean better separation, but can favor over-separated graphs.
# dataset/condition silhouette: very high values suggest remaining dataset/condition separation in latent space.

try:
    from sklearn.metrics import silhouette_score

    rng = np.random.default_rng(RANDOM_STATE)
    n_sub = min(SILHOUETTE_MAX_CELLS, adata.n_obs)
    sub_idx = rng.choice(adata.n_obs, size=n_sub, replace=False)
    X_sub = np.asarray(adata.obsm[EMBED_KEY][sub_idx, :], dtype=np.float32)

    sil_rows = []

    # Dataset/condition silhouette on latent space
    for label_col in ["core_dataset", "condition"]:
        labels = adata.obs.iloc[sub_idx][label_col].astype(str).values
        if len(np.unique(labels)) > 1:
            sil = silhouette_score(X_sub, labels, metric="euclidean")
        else:
            sil = np.nan
        sil_rows.append({
            "resolution": "NA",
            "cluster_key": "NA",
            "metric": f"{label_col}_silhouette_on_X_scVI_subsample",
            "n_cells_subsample": int(n_sub),
            "value": float(sil) if pd.notna(sil) else np.nan,
            "interpretation": "High value may indicate separation by metadata label; lower/near-zero usually indicates better mixing for technical labels.",
        })

    # Cluster silhouette per resolution
    for _, row in sweep_df.iterrows():
        key = row["cluster_key"]
        labels = adata.obs.iloc[sub_idx][key].astype(str).values
        if len(np.unique(labels)) > 1 and len(np.unique(labels)) < len(labels):
            sil = silhouette_score(X_sub, labels, metric="euclidean")
        else:
            sil = np.nan

        sil_rows.append({
            "resolution": float(row["resolution"]),
            "cluster_key": key,
            "metric": "cluster_silhouette_on_X_scVI_subsample",
            "n_cells_subsample": int(n_sub),
            "value": float(sil) if pd.notna(sil) else np.nan,
            "interpretation": "Higher can indicate sharper clustering, but it must be interpreted with marker and bias QC.",
        })

    silhouette_df = pd.DataFrame(sil_rows)
    write_tsv_replace(silhouette_df, SILHOUETTE_TSV)
    display(silhouette_df)

except Exception as e:
    silhouette_df = pd.DataFrame([{
        "error": repr(e),
        "note": "Silhouette metrics failed or sklearn is unavailable. The rest of the notebook can continue."
    }])
    write_tsv_replace(silhouette_df, SILHOUETTE_TSV)
    print("[WARNING] Silhouette calculation skipped:", repr(e))

[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\clustering_resolution_silhouette_metrics_HVG8000_rerun_lowmem.tsv shape=(18, 6)


,resolution,cluster_key,metric,n_cells_subsample,value,interpretation
0,NA,NA,core_dataset_silhouette_on_X_scVI_subsample,30000,-0.050770,High value may indicate separation by metadata...
1,NA,NA,condition_silhouette_on_X_scVI_subsample,30000,-0.026369,High value may indicate separation by metadata...
2,0.1,leiden_scvi_HVG8000_r0_1,cluster_silhouette_on_X_scVI_subsample,30000,0.032692,"Higher can indicate sharper clustering, but it..."
3,0.15,leiden_scvi_HVG8000_r0_15,cluster_silhouette_on_X_scVI_subsample,30000,0.032503,"Higher can indicate sharper clustering, but it..."
4,0.2,leiden_scvi_HVG8000_r0_2,cluster_silhouette_on_X_scVI_subsample,30000,0.044264,"Higher can indicate sharper clustering, but it..."
5,0.25,leiden_scvi_HVG8000_r0_25,cluster_silhouette_on_X_scVI_subsample,30000,0.052037,"Higher can indicate sharper clustering, but it..."
6,0.3,leiden_scvi_HVG8000_r0_3,cluster_silhouette_on_X_scVI_subsample,30000,0.053284,"Higher can indicate sharper clustering, but it..."
7,0.35,leiden_scvi_HVG8000_r0_35,cluster_silhouette_on_X_scVI_subsample,30000,0.062465,"Higher can indicate sharper clustering, but it..."
8,0.4,leiden_scvi_HVG8000_r0_4,cluster_silhouette_on_X_scVI_subsample,30000,0.061082,"Higher can indicate sharper clustering, but it..."
9,0.45,leiden_scvi_HVG8000_r0_45,cluster_silhouette_on_X_scVI_subsample,30000,0.062032,"Higher can indicate sharper clustering, but it..."


In [7]:
# =========================
# 5) BASIC UMAP FIGURES FOR METADATA AND RESOLUTIONS
# =========================

# Compute UMAP if missing.
if UMAP_KEY not in adata.obsm.keys():
    print("X_umap missing. Computing UMAP from scVI neighbors.")
    sc.tl.umap(adata, neighbors_key=NEIGHBORS_KEY, random_state=RANDOM_STATE, min_dist=0.3)
else:
    print("X_umap already present.")

# UMAP colored by metadata
metadata_fig_path = FIGURES_DIR / "clustering_QC_HVG8000_UMAP_metadata_rerun_lowmem.png"
with plt.rc_context({"figure.figsize": (6, 5)}):
    sc.pl.umap(
        adata,
        color=["core_dataset", "condition"],
        wspace=0.4,
        show=False,
    )
    plt.savefig(metadata_fig_path, dpi=180, bbox_inches="tight")
    plt.close()
print("[SAVED]", metadata_fig_path)

# UMAP for all sweep resolutions in batches to avoid giant figures.
resolution_keys = list(sweep_df["cluster_key"])

batch_size = 4
for i in range(0, len(resolution_keys), batch_size):
    keys = resolution_keys[i:i+batch_size]
    fig_path = FIGURES_DIR / f"clustering_QC_HVG8000_UMAP_resolution_rerun_batch_{i//batch_size+1}_lowmem.png"
    with plt.rc_context({"figure.figsize": (5 * len(keys), 4.5)}):
        sc.pl.umap(
            adata,
            color=keys,
            wspace=0.35,
            legend_loc="on data",
            legend_fontsize=6,
            show=False,
        )
        plt.savefig(fig_path, dpi=170, bbox_inches="tight")
        plt.close()
    print("[SAVED]", fig_path)

X_umap already present.
[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\figures\clustering_QC_HVG8000_UMAP_metadata_rerun_lowmem.png
[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\figures\clustering_QC_HVG8000_UMAP_resolution_rerun_batch_1_lowmem.png
[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\figures\clustering_QC_HVG8000_UMAP_resolution_rerun_batch_2_lowmem.png
[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\figures\clustering_QC_HVG8000_UMAP_resolution_rerun_batch_3_lowmem.png
[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\figures\clustering_QC_HVG8000_UMAP_resolution_rerun_batch_4_lowmem.png


In [8]:
# =========================
# 6) LOAD FULL-GENE MARKER PANEL SUBSET FOR BIOLOGICAL QC
# =========================

# We use the full-gene object only for a small marker-gene subset.
# This avoids using the HVG-only matrix for marker dotplots, where some important markers may be absent.

marker_panels = {
    "Tumor_AC_like": ["GFAP", "AQP4", "ALDH1L1", "SLC1A3", "CLU", "APOE"],
    "Tumor_OPC_like": ["PDGFRA", "OLIG1", "OLIG2", "SOX10", "CSPG4", "BCAN"],
    "Tumor_NPC_like": ["SOX2", "NES", "VIM", "FABP7", "DLL3", "ASCL1"],
    "Tumor_MES_like": ["CHI3L1", "CD44", "ANXA1", "VIM", "SERPINE1", "TNC", "EMP3"],
    "Cycling": ["MKI67", "TOP2A", "PCNA", "UBE2C", "STMN1", "HMGB2"],
    "Hypoxia_stress": ["VEGFA", "CA9", "NDRG1", "DDIT4", "HILPDA", "ENO1"],
    "Myeloid": ["PTPRC", "LYZ", "C1QA", "C1QB", "C1QC", "TYROBP", "AIF1", "CSF1R"],
    "Inflammatory_myeloid": ["IL1B", "CCL3", "CCL4", "CXCL8", "NFKBIA", "TNF"],
    "T_NK": ["CD3D", "CD3E", "TRAC", "IL7R", "NKG7", "GNLY", "GZMB"],
    "B_Plasma": ["MS4A1", "CD79A", "CD79B", "MZB1", "JCHAIN", "SDC1"],
    "Oligodendrocyte": ["MBP", "PLP1", "MOG", "MAG", "MOBP", "CLDN11"],
    "Endothelial": ["PECAM1", "VWF", "KDR", "FLT1", "CLDN5", "RAMP2"],
    "Mural_Pericyte": ["PDGFRB", "RGS5", "ACTA2", "TAGLN", "MCAM", "CSPG4"],
    "Fibroblast_ECM": ["COL1A1", "COL1A2", "DCN", "LUM", "COL3A1", "FAP"],
}

all_marker_genes = sorted(set(g for genes in marker_panels.values() for g in genes))

if not FULLGENE_H5AD.exists():
    print(f"[WARNING] Full-gene h5ad not found: {FULLGENE_H5AD}")
    print("Marker module scoring and dotplots will be skipped.")
    adata_marker = None
else:
    print("Reading marker subset from full-gene object:", FULLGENE_H5AD)
    adata_full_backed = sc.read_h5ad(FULLGENE_H5AD, backed="r")

    present_genes = [g for g in all_marker_genes if g in adata_full_backed.var_names]
    missing_genes = [g for g in all_marker_genes if g not in adata_full_backed.var_names]

    marker_presence = pd.DataFrame({
        "gene": all_marker_genes,
        "present_in_fullgene": [g in present_genes for g in all_marker_genes],
    })
    MARKER_PRESENCE_TSV = METADATA_DIR / "clustering_resolution_marker_panel_presence_HVG8000_rerun_lowmem.tsv"
    write_tsv_replace(marker_presence, MARKER_PRESENCE_TSV)

    print("Marker genes present:", len(present_genes), "/", len(all_marker_genes))
    if missing_genes:
        print("Missing marker genes:", missing_genes)

    if len(present_genes) == 0:
        raise ValueError("None of the marker genes were found in the full-gene object.")

    # Subset only marker genes into memory.
    adata_marker = adata_full_backed[:, present_genes].to_memory()
    adata_full_backed.file.close()
    del adata_full_backed

    # Align cells and transfer only obs columns from the lightweight scVI
    # object. Do NOT copy adata itself, because that would duplicate graph
    # matrices and embeddings unnecessarily.
    common = adata_marker.obs_names.intersection(adata.obs_names)
    print("Common cells marker/fullgene vs scVI:", len(common), "/", adata.n_obs)

    if len(common) != adata.n_obs:
        raise AssertionError(
            "Full-gene marker object and scVI object do not contain the "
            "same cell set."
        )

    if not adata_marker.obs_names.equals(adata.obs_names):
        adata_marker = adata_marker[adata.obs_names].copy()

    transfer_cols = (
        ["core_dataset", "condition", "patient_id", "sample_id", "scvi_batch"]
        + resolution_keys
    )

    for col in transfer_cols:
        if col in adata.obs.columns:
            adata_marker.obs[col] = (
                adata.obs.loc[adata_marker.obs_names, col]
                .astype(str)
                .to_numpy()
            )

    # Use raw counts for the small marker subset without retaining a
    # duplicate counts layer in memory.
    if "counts" in adata_marker.layers:
        adata_marker.X = adata_marker.layers["counts"]
        del adata_marker.layers["counts"]

    sc.pp.normalize_total(
        adata_marker,
        target_sum=1e4,
    )
    sc.pp.log1p(adata_marker)

    print(adata_marker)
    gc.collect()

Reading marker subset from full-gene object: G:\Repository\glioma-cnv-single-cell-atlas\data\processed\integrated\GBM_core_4datasets_fullgene_postQC_merged_counts_metadata.h5ad
[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\clustering_resolution_marker_panel_presence_HVG8000_rerun_lowmem.tsv shape=(86, 2)
Marker genes present: 84 / 86
Missing marker genes: ['JCHAIN', 'MS4A1']
Common cells marker/fullgene vs scVI: 351427 / 351427
normalizing counts per cell
    finished (0:00:01)
AnnData object with n_obs × n_vars = 351427 × 84
    obs: 'sample_id', 'geo_id', 'core_dataset', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'total_counts_ribo', 'pct_counts_ribo', 'total_counts_hb', 'pct_counts_hb', 'dataset_id', 'study_id', 'source_dataset', 'source_file', 'concat_key', 'patient_id', 'condition', 'batch_dataset', 'batch_sample', 'batch_patient', 'original_cell_barcode', 'diagnosis', 'disease_stage', 'tumor_status', 'platform', 'technology', 'study_bat

In [9]:
# =========================
# 7) MARKER MODULE SCORING PER RESOLUTION
# =========================

if adata_marker is None:
    marker_module_df = pd.DataFrame([{
        "note": "Marker module scoring skipped because full-gene marker subset was unavailable."
    }])
    write_tsv_replace(marker_module_df, MARKER_MODULE_TSV)

else:
    # Score marker modules using present genes only.
    present_marker_panels = {}
    for module, genes in marker_panels.items():
        present = [g for g in genes if g in adata_marker.var_names]
        if len(present) >= 2:
            present_marker_panels[module] = present

    print("Marker modules with >=2 present genes:")
    for m, genes in present_marker_panels.items():
        print(m, genes)

    for module, genes in present_marker_panels.items():
        score_col = f"score_{module}"
        sc.tl.score_genes(
            adata_marker,
            gene_list=genes,
            score_name=score_col,
            use_raw=False,
            random_state=RANDOM_STATE,
        )

    score_cols = [f"score_{m}" for m in present_marker_panels.keys()]

    module_rows = []
    for _, row in sweep_df.iterrows():
        res = float(row["resolution"])
        key = row["cluster_key"]

        cluster_means = adata_marker.obs.groupby(key, observed=True)[score_cols].mean()

        for cl, vals in cluster_means.iterrows():
            sorted_vals = vals.sort_values(ascending=False)
            top_module = sorted_vals.index[0].replace("score_", "")
            second_module = sorted_vals.index[1].replace("score_", "") if len(sorted_vals) > 1 else "NA"
            top_score = float(sorted_vals.iloc[0])
            second_score = float(sorted_vals.iloc[1]) if len(sorted_vals) > 1 else np.nan

            module_rows.append({
                "resolution": res,
                "cluster_key": key,
                "cluster": str(cl),
                "top_marker_module": top_module,
                "second_marker_module": second_module,
                "top_module_score": top_score,
                "second_module_score": second_score,
                "top_minus_second_module_score": float(top_score - second_score) if pd.notna(second_score) else np.nan,
            })

    marker_module_df = pd.DataFrame(module_rows)
    write_tsv_replace(marker_module_df, MARKER_MODULE_TSV)
    display(marker_module_df.head())

Marker modules with >=2 present genes:
Tumor_AC_like ['GFAP', 'AQP4', 'ALDH1L1', 'SLC1A3', 'CLU', 'APOE']
Tumor_OPC_like ['PDGFRA', 'OLIG1', 'OLIG2', 'SOX10', 'CSPG4', 'BCAN']
Tumor_NPC_like ['SOX2', 'NES', 'VIM', 'FABP7', 'DLL3', 'ASCL1']
Tumor_MES_like ['CHI3L1', 'CD44', 'ANXA1', 'VIM', 'SERPINE1', 'TNC', 'EMP3']
Cycling ['MKI67', 'TOP2A', 'PCNA', 'UBE2C', 'STMN1', 'HMGB2']
Hypoxia_stress ['VEGFA', 'CA9', 'NDRG1', 'DDIT4', 'HILPDA', 'ENO1']
Myeloid ['PTPRC', 'LYZ', 'C1QA', 'C1QB', 'C1QC', 'TYROBP', 'AIF1', 'CSF1R']
Inflammatory_myeloid ['IL1B', 'CCL3', 'CCL4', 'CXCL8', 'NFKBIA', 'TNF']
T_NK ['CD3D', 'CD3E', 'TRAC', 'IL7R', 'NKG7', 'GNLY', 'GZMB']
B_Plasma ['CD79A', 'CD79B', 'MZB1', 'SDC1']
Oligodendrocyte ['MBP', 'PLP1', 'MOG', 'MAG', 'MOBP', 'CLDN11']
Endothelial ['PECAM1', 'VWF', 'KDR', 'FLT1', 'CLDN5', 'RAMP2']
Mural_Pericyte ['PDGFRB', 'RGS5', 'ACTA2', 'TAGLN', 'MCAM', 'CSPG4']
Fibroblast_ECM ['COL1A1', 'COL1A2', 'DCN', 'LUM', 'COL3A1', 'FAP']
computing score 'score_Tumor_AC_like

,resolution,cluster_key,cluster,top_marker_module,second_marker_module,top_module_score,second_module_score,top_minus_second_module_score
0,0.1,leiden_scvi_HVG8000_r0_1,0,Tumor_NPC_like,Tumor_AC_like,1.684855,1.377502,0.307353
1,0.1,leiden_scvi_HVG8000_r0_1,1,Myeloid,Tumor_MES_like,3.107488,0.850661,2.256827
2,0.1,leiden_scvi_HVG8000_r0_1,2,T_NK,Tumor_MES_like,3.233121,1.551667,1.681454
3,0.1,leiden_scvi_HVG8000_r0_1,3,Oligodendrocyte,Tumor_OPC_like,5.160623,1.216938,3.943685
4,0.1,leiden_scvi_HVG8000_r0_1,4,Tumor_MES_like,Fibroblast_ECM,1.731324,1.653302,0.078023


In [10]:
# =========================
# 8) BUILD ADVISOR TABLE FOR RESOLUTION SELECTION
# =========================

# This scoring is only an advisor. It should not be treated as an automatic final truth.
# It balances cluster granularity, small-cluster burden, metadata bias, silhouette, and marker module separation.

advisor = sweep_df.merge(
    composition_summary_df,
    on=["resolution", "cluster_key", "n_clusters"],
    how="left"
)

# Add cluster silhouette
if "metric" in silhouette_df.columns:
    cluster_sil = silhouette_df[
        silhouette_df["metric"] == "cluster_silhouette_on_X_scVI_subsample"
    ][["resolution", "cluster_key", "value"]].rename(columns={"value": "cluster_silhouette_subsample"})
    advisor = advisor.merge(cluster_sil, on=["resolution", "cluster_key"], how="left")
else:
    advisor["cluster_silhouette_subsample"] = np.nan

# Add marker module summary
if "top_minus_second_module_score" in marker_module_df.columns:
    marker_summary = marker_module_df.groupby(["resolution", "cluster_key"], observed=True).agg(
        median_top_minus_second_module_score=("top_minus_second_module_score", "median"),
        mean_top_minus_second_module_score=("top_minus_second_module_score", "mean"),
        fraction_clusters_marker_separation_gt_0_10=("top_minus_second_module_score", lambda x: float(np.mean(np.asarray(x) > 0.10))),
        fraction_clusters_marker_separation_gt_0_20=("top_minus_second_module_score", lambda x: float(np.mean(np.asarray(x) > 0.20))),
    ).reset_index()
    advisor = advisor.merge(marker_summary, on=["resolution", "cluster_key"], how="left")
else:
    advisor["median_top_minus_second_module_score"] = np.nan
    advisor["mean_top_minus_second_module_score"] = np.nan
    advisor["fraction_clusters_marker_separation_gt_0_10"] = np.nan
    advisor["fraction_clusters_marker_separation_gt_0_20"] = np.nan

# -------------------------
# Scoring components
# -------------------------

def cluster_count_score(n):
    # Broad preference: enough granularity to separate GBM lineages/states, not too fragmented.
    # This is NOT targeting a fixed number like 24.
    if 15 <= n <= 35:
        return 1.0
    if 10 <= n < 15:
        return 0.75
    if 35 < n <= 45:
        return 0.75
    if 8 <= n < 10:
        return 0.45
    if 45 < n <= 60:
        return 0.45
    return 0.20

advisor["score_cluster_count_range"] = advisor["n_clusters"].apply(cluster_count_score)

# Penalize many cells in tiny clusters.
advisor["score_small_cluster_burden"] = 1.0 - np.minimum(1.0, advisor["frac_cells_in_clusters_lt_500"] * 10.0)

# Penalize very strong dataset dominance. Not all dataset structure is technical, so weight is moderate.
advisor["score_dataset_mixing"] = 1.0 - np.clip(
    (advisor["mean_top_dataset_fraction_weighted"] - 0.55) / 0.45,
    0,
    1,
)

# Penalize extreme patient/sample dominance. Some patient-specific tumor states can be real, so this is moderate.
advisor["score_patient_not_overdominant"] = 1.0 - np.clip(
    (advisor["mean_top_patient_fraction_weighted"] - 0.35) / 0.65,
    0,
    1,
)

# Prefer clear marker-module separation when available.
if "median_top_minus_second_module_score" in advisor.columns:
    max_marker_sep = np.nanmax(advisor["median_top_minus_second_module_score"].values)
    min_marker_sep = np.nanmin(advisor["median_top_minus_second_module_score"].values)
    denom = max(max_marker_sep - min_marker_sep, 1e-9)
    advisor["score_marker_module_separation"] = (
        advisor["median_top_minus_second_module_score"] - min_marker_sep
    ) / denom
else:
    advisor["score_marker_module_separation"] = np.nan

advisor["score_marker_module_separation"] = advisor["score_marker_module_separation"].fillna(0.5)

# Cluster silhouette: normalize across tested resolutions.
if "cluster_silhouette_subsample" in advisor.columns and advisor["cluster_silhouette_subsample"].notna().any():
    smin = advisor["cluster_silhouette_subsample"].min()
    smax = advisor["cluster_silhouette_subsample"].max()
    denom = max(smax - smin, 1e-9)
    advisor["score_cluster_silhouette_relative"] = (advisor["cluster_silhouette_subsample"] - smin) / denom
else:
    advisor["score_cluster_silhouette_relative"] = 0.5

# Final advisor score with transparent weights.
advisor["advisor_score"] = (
    0.20 * advisor["score_cluster_count_range"] +
    0.20 * advisor["score_small_cluster_burden"] +
    0.15 * advisor["score_dataset_mixing"] +
    0.10 * advisor["score_patient_not_overdominant"] +
    0.25 * advisor["score_marker_module_separation"] +
    0.10 * advisor["score_cluster_silhouette_relative"]
)

# Add human-readable flags.
flags = []
for _, r in advisor.iterrows():
    fs = []
    if r["n_clusters"] < 10:
        fs.append("very_coarse")
    if r["n_clusters"] > 45:
        fs.append("possibly_overclustered")
    if r["frac_cells_in_clusters_lt_500"] > 0.01:
        fs.append("many_cells_in_tiny_clusters")
    if r.get("mean_top_dataset_fraction_weighted", 0) > 0.75:
        fs.append("dataset_dominance_risk")
    if r.get("mean_top_patient_fraction_weighted", 0) > 0.55:
        fs.append("patient_dominance_risk")
    if r.get("median_top_minus_second_module_score", np.nan) < 0.05:
        fs.append("weak_marker_module_separation")
    flags.append(";".join(fs) if fs else "no_major_flag")

advisor["advisor_flags"] = flags

advisor = advisor.sort_values("advisor_score", ascending=False).reset_index(drop=True)
write_tsv_replace(advisor, ADVISOR_TSV)

display(advisor[[
    "resolution", "cluster_key", "n_clusters",
    "min_cluster_size", "frac_cells_in_clusters_lt_500",
    "mean_top_dataset_fraction_weighted",
    "mean_top_patient_fraction_weighted",
    "median_top_minus_second_module_score",
    "cluster_silhouette_subsample",
    "advisor_score",
    "advisor_flags",
]].head(16))

recommendation = advisor.head(5).copy()
recommendation["note"] = (
    "Top candidate resolutions by advisor_score. Final choice should also inspect UMAP, marker dotplots, composition heatmaps, and biological interpretability."
)
write_tsv_replace(recommendation, RECOMMENDATION_TSV)
display(recommendation)

# -------------------------
# Fresh-rerun advisor status vs archived article selection
# -------------------------

current_top_resolution = float(
    advisor.iloc[0]["resolution"]
)

fresh_rerun_advisor_ranks_article_resolution_first = bool(
    np.isclose(
        current_top_resolution,
        ARTICLE_SELECTED_RESOLUTION,
    )
)

status_df[
    "fresh_rerun_advisor_rank_1_resolution"
] = current_top_resolution

status_df[
    "fresh_rerun_advisor_ranks_article_resolution_first"
] = fresh_rerun_advisor_ranks_article_resolution_first

write_tsv_replace(
    status_df,
    STATUS_TSV,
)

display(
    status_df
)

print(
    "Fresh-rerun advisor rank #1:",
    current_top_resolution,
)

print(
    "Archived article-selected resolution:",
    ARTICLE_SELECTED_RESOLUTION,
)

if not fresh_rerun_advisor_ranks_article_resolution_first:
    print(
        "[AUDIT NOTE] The fresh-rerun advisor does not rank "
        "the archived article-selected resolution first. "
        "This does not redefine the article selection."
    )

[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\clustering_resolution_advisor_table_HVG8000_rerun_lowmem.tsv shape=(16, 43)


,resolution,cluster_key,n_clusters,min_cluster_size,frac_cells_in_clusters_lt_500,mean_top_dataset_fraction_weighted,mean_top_patient_fraction_weighted,median_top_minus_second_module_score,cluster_silhouette_subsample,advisor_score,advisor_flags
0,0.6,leiden_scvi_HVG8000_r0_6,15,586,0.000000,0.632854,0.243385,1.431249,0.060246,0.855412,no_major_flag
1,0.55,leiden_scvi_HVG8000_r0_55,16,1380,0.000000,0.633369,0.242656,1.416050,0.061247,0.853208,no_major_flag
2,0.45,leiden_scvi_HVG8000_r0_45,15,189,0.000538,0.630524,0.240428,1.394541,0.062032,0.849529,no_major_flag
3,0.4,leiden_scvi_HVG8000_r0_4,9,2653,0.000000,0.629454,0.236647,1.670852,0.061082,0.796338,very_coarse
4,0.25,leiden_scvi_HVG8000_r0_25,9,1179,0.000000,0.629428,0.228955,1.674978,0.052037,0.787528,very_coarse
5,0.8,leiden_scvi_HVG8000_r0_8,22,187,0.000532,0.632672,0.279350,1.503266,-0.024360,0.778739,no_major_flag
6,0.1,leiden_scvi_HVG8000_r0_1,6,1145,0.000000,0.629456,0.210533,1.961732,0.032692,0.775378,very_coarse
7,0.15,leiden_scvi_HVG8000_r0_15,6,1146,0.000000,0.629343,0.210601,1.960042,0.032503,0.774870,very_coarse
8,1.0,leiden_scvi_HVG8000_r1_0,24,169,0.000481,0.631343,0.282212,1.431769,-0.015398,0.774265,no_major_flag
9,0.9,leiden_scvi_HVG8000_r0_9,23,95,0.000270,0.632791,0.277799,1.242715,0.004752,0.757142,no_major_flag


[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\clustering_resolution_recommended_resolution_HVG8000_rerun_lowmem.tsv shape=(5, 44)


,resolution,cluster_key,n_clusters,min_cluster_size,max_cluster_size,median_cluster_size,mean_cluster_size,n_clusters_lt_50_cells,n_clusters_lt_100_cells,n_clusters_lt_500_cells,...,fraction_clusters_marker_separation_gt_0_20,score_cluster_count_range,score_small_cluster_burden,score_dataset_mixing,score_patient_not_overdominant,score_marker_module_separation,score_cluster_silhouette_relative,advisor_score,advisor_flags,note
0,0.6,leiden_scvi_HVG8000_r0_6,15,586,71112,21853.0,23428.466667,0,0,0,...,0.933333,1.00,1.000000,0.815880,1.0,0.566944,0.912945,0.855412,no_major_flag,Top candidate resolutions by advisor_score. Fi...
1,0.55,leiden_scvi_HVG8000_r0_55,16,1380,72364,12440.5,21964.187500,0,0,0,...,0.812500,1.00,1.000000,0.814735,1.0,0.554536,0.923638,0.853208,no_major_flag,Top candidate resolutions by advisor_score. Fi...
2,0.45,leiden_scvi_HVG8000_r0_45,15,189,97771,14933.0,23428.466667,0,0,1,...,0.933333,1.00,0.994622,0.821059,1.0,0.536977,0.932019,0.849529,no_major_flag,Top candidate resolutions by advisor_score. Fi...
3,0.4,leiden_scvi_HVG8000_r0_4,9,2653,131539,21762.0,39047.444444,0,0,0,...,0.888889,0.45,1.000000,0.823436,1.0,0.762542,0.921869,0.796338,very_coarse,Top candidate resolutions by advisor_score. Fi...
4,0.25,leiden_scvi_HVG8000_r0_25,9,1179,129955,14913.0,39047.444444,0,0,0,...,0.888889,0.45,1.000000,0.823493,1.0,0.765910,0.825263,0.787528,very_coarse,Top candidate resolutions by advisor_score. Fi...


[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\clustering_resolution_reproducibility_status_HVG8000.tsv shape=(1, 14)


,article_selected_resolution,historical_n_clusters,fresh_rerun_n_clusters,historical_min_cluster_size,fresh_rerun_min_cluster_size,historical_max_cluster_size,fresh_rerun_max_cluster_size,historical_median_cluster_size,fresh_rerun_median_cluster_size,fresh_rerun_reproduces_archived_cluster_count,fresh_rerun_reproduces_archived_size_summary,article_downstream_source,fresh_rerun_advisor_rank_1_resolution,fresh_rerun_advisor_ranks_article_resolution_first
0,0.6,18,15,105,586,61646,71112,14336.5,21853.0,False,False,canonical_historical_artifact_required,0.6,True


Fresh-rerun advisor rank #1: 0.6
Archived article-selected resolution: 0.6


In [11]:
# =========================
# 9) DETAILED FIGURES FOR TOP CANDIDATE RESOLUTIONS
# =========================

top_candidate_keys = list(dict.fromkeys(advisor.head(N_TOP_CANDIDATE_FIGURES)["cluster_key"].tolist()))

# Always include common reference resolutions if available, but do not force them as final.
for ref_res in [0.30, 0.40, 0.50, 0.60]:
    ref_key = f"leiden_scvi_HVG8000_r{str(ref_res).replace('.', '_')}"
    if ref_key in adata.obs.columns and ref_key not in top_candidate_keys:
        top_candidate_keys.append(ref_key)

print("Detailed figure cluster keys:")
print(top_candidate_keys)

# UMAP detailed for candidates
for key in top_candidate_keys:
    fig_path = FIGURES_DIR / f"clustering_QC_HVG8000_rerun_UMAP_{key}_lowmem.png"
    with plt.rc_context({"figure.figsize": (7, 6)}):
        sc.pl.umap(
            adata,
            color=key,
            legend_loc="on data",
            legend_fontsize=6,
            show=False,
        )
        plt.savefig(fig_path, dpi=220, bbox_inches="tight")
        plt.close()
    print("[SAVED]", fig_path)

# Composition heatmaps for candidates
for key in top_candidate_keys:
    comp = composition_df[composition_df["cluster_key"] == key].copy()
    if comp.empty:
        continue

    # Dataset composition
    tab_dataset = pd.crosstab(adata.obs[key].astype(str), adata.obs["core_dataset"].astype(str), normalize="index")
    tab_condition = pd.crosstab(adata.obs[key].astype(str), adata.obs["condition"].astype(str), normalize="index")

    for tab, label in [(tab_dataset, "dataset"), (tab_condition, "condition")]:
        # Sort cluster index numerically when possible
        try:
            tab = tab.loc[sorted(tab.index, key=lambda x: int(x) if str(x).isdigit() else str(x))]
        except Exception:
            pass

        fig_h = max(4, 0.25 * tab.shape[0])
        fig_w = max(6, 0.7 * tab.shape[1] + 4)
        fig, ax = plt.subplots(figsize=(fig_w, fig_h))
        im = ax.imshow(tab.values, aspect="auto", vmin=0, vmax=1)
        ax.set_yticks(np.arange(tab.shape[0]))
        ax.set_yticklabels(tab.index, fontsize=7)
        ax.set_xticks(np.arange(tab.shape[1]))
        ax.set_xticklabels(tab.columns, rotation=45, ha="right", fontsize=8)
        ax.set_title(f"{label} composition per cluster — {key}")
        cbar = plt.colorbar(im, ax=ax)
        cbar.set_label("Fraction within cluster")
        plt.tight_layout()
        fig_path = FIGURES_DIR / f"clustering_QC_HVG8000_heatmap_{label}_composition_{key}_lowmem.png"
        plt.savefig(fig_path, dpi=220, bbox_inches="tight")
        plt.close()
        print("[SAVED]", fig_path)

# Marker dotplots and module heatmaps
if adata_marker is not None:
    present_marker_panels_for_dotplot = {
        module: [g for g in genes if g in adata_marker.var_names]
        for module, genes in marker_panels.items()
    }
    present_marker_panels_for_dotplot = {
        module: genes for module, genes in present_marker_panels_for_dotplot.items()
        if len(genes) > 0
    }

    for key in top_candidate_keys:
        if key not in adata_marker.obs.columns:
            continue

        # Dotplot
        try:
            dot_path = FIGURES_DIR / f"clustering_QC_HVG8000_rerun_marker_dotplot_{key}_lowmem.png"
            dp = sc.pl.dotplot(
                adata_marker,
                var_names=present_marker_panels_for_dotplot,
                groupby=key,
                standard_scale="var",
                dendrogram=False,
                show=False,
                return_fig=True,
            )
            dp.savefig(dot_path)
            plt.close("all")
            print("[SAVED]", dot_path)
        except Exception as e:
            print(f"[WARNING] Dotplot failed for {key}:", repr(e))

        # Marker module score heatmap
        sub = marker_module_df[marker_module_df["cluster_key"] == key].copy()
        if sub.empty:
            continue

        # Build matrix: clusters x marker modules from adata_marker obs score_cols
        score_cols = [c for c in adata_marker.obs.columns if c.startswith("score_")]
        module_mean = adata_marker.obs.groupby(key, observed=True)[score_cols].mean()
        module_mean.columns = [c.replace("score_", "") for c in module_mean.columns]

        try:
            module_mean = module_mean.loc[sorted(module_mean.index, key=lambda x: int(x) if str(x).isdigit() else str(x))]
        except Exception:
            pass

        # z-score per module for visualization
        mat = module_mean.copy()
        mat = (mat - mat.mean(axis=0)) / (mat.std(axis=0) + 1e-9)

        fig_h = max(5, 0.25 * mat.shape[0])
        fig_w = max(8, 0.45 * mat.shape[1])
        fig, ax = plt.subplots(figsize=(fig_w, fig_h))
        im = ax.imshow(mat.values, aspect="auto", vmin=-2, vmax=2)
        ax.set_yticks(np.arange(mat.shape[0]))
        ax.set_yticklabels(mat.index, fontsize=7)
        ax.set_xticks(np.arange(mat.shape[1]))
        ax.set_xticklabels(mat.columns, rotation=60, ha="right", fontsize=8)
        ax.set_title(f"Marker module score heatmap — {key}")
        cbar = plt.colorbar(im, ax=ax)
        cbar.set_label("Z-scored mean module score")
        plt.tight_layout()
        fig_path = FIGURES_DIR / f"clustering_QC_HVG8000_marker_module_heatmap_{key}_lowmem.png"
        plt.savefig(fig_path, dpi=220, bbox_inches="tight")
        plt.close()
        print("[SAVED]", fig_path)

else:
    print("Skipping marker dotplots because adata_marker is unavailable.")

Detailed figure cluster keys:
['leiden_scvi_HVG8000_r0_6', 'leiden_scvi_HVG8000_r0_55', 'leiden_scvi_HVG8000_r0_45', 'leiden_scvi_HVG8000_r0_4', 'leiden_scvi_HVG8000_r0_3', 'leiden_scvi_HVG8000_r0_5']
[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\figures\clustering_QC_HVG8000_rerun_UMAP_leiden_scvi_HVG8000_r0_6_lowmem.png
[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\figures\clustering_QC_HVG8000_rerun_UMAP_leiden_scvi_HVG8000_r0_55_lowmem.png
[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\figures\clustering_QC_HVG8000_rerun_UMAP_leiden_scvi_HVG8000_r0_45_lowmem.png
[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\figures\clustering_QC_HVG8000_rerun_UMAP_leiden_scvi_HVG8000_r0_4_lowmem.png
[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\figures\clustering_QC_HVG8000_rerun_UMAP_leiden_scvi_HVG8000_r0_3_lowmem.png
[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\figures\clustering_QC_HVG8000_rerun_UMAP_leiden_scvi_HVG8000_r0_5_lowmem.png
[SAVED] G:\Repository\glioma-

In [12]:
# =========================
# 10) SAVE FRESH-RERUN H5AD + MANIFEST
# =========================

# Publish a fresh-rerun artifact only. Its filename explicitly contains
# "rerun" so it cannot be confused with the article-defining historical
# 18-cluster artifact.

write_h5ad_replace(
    adata,
    OUT_H5AD,
)


manifest_rows = [
    {
        "name": "input_scvi_h5ad",
        "path": str(SCVI_H5AD),
        "exists": SCVI_H5AD.exists(),
    },
    {
        "name": "input_hvg_counts_h5ad",
        "path": str(HVG_COUNTS_H5AD),
        "exists": HVG_COUNTS_H5AD.exists(),
    },
    {
        "name": "fullgene_h5ad",
        "path": str(FULLGENE_H5AD),
        "exists": FULLGENE_H5AD.exists(),
    },
    {
        "name": "scratch_checkpoint_h5ad",
        "path": str(CHECKPOINT_H5AD),
        "exists": CHECKPOINT_H5AD.exists(),
    },
    {
        "name": "fresh_rerun_resolution_qc_h5ad",
        "path": str(OUT_H5AD),
        "exists": OUT_H5AD.exists(),
    },
    {
        "name": "input_qc_tsv",
        "path": str(INPUT_QC_TSV),
        "exists": INPUT_QC_TSV.exists(),
    },
    {
        "name": "sweep_summary_tsv",
        "path": str(SWEEP_SUMMARY_TSV),
        "exists": SWEEP_SUMMARY_TSV.exists(),
    },
    {
        "name": "cluster_sizes_tsv",
        "path": str(CLUSTER_SIZES_TSV),
        "exists": CLUSTER_SIZES_TSV.exists(),
    },
    {
        "name": "composition_tsv",
        "path": str(COMPOSITION_TSV),
        "exists": COMPOSITION_TSV.exists(),
    },
    {
        "name": "composition_summary_tsv",
        "path": str(COMPOSITION_SUMMARY_TSV),
        "exists": COMPOSITION_SUMMARY_TSV.exists(),
    },
    {
        "name": "silhouette_tsv",
        "path": str(SILHOUETTE_TSV),
        "exists": SILHOUETTE_TSV.exists(),
    },
    {
        "name": "marker_module_tsv",
        "path": str(MARKER_MODULE_TSV),
        "exists": MARKER_MODULE_TSV.exists(),
    },
    {
        "name": "advisor_tsv",
        "path": str(ADVISOR_TSV),
        "exists": ADVISOR_TSV.exists(),
    },
    {
        "name": "recommendation_tsv",
        "path": str(RECOMMENDATION_TSV),
        "exists": RECOMMENDATION_TSV.exists(),
    },
    {
        "name": "reproducibility_status_tsv",
        "path": str(STATUS_TSV),
        "exists": STATUS_TSV.exists(),
    },
]


manifest = pd.DataFrame(
    manifest_rows
)

write_tsv_replace(
    manifest,
    MANIFEST_TSV,
)

display(
    manifest
)


print(
    "\nDONE — notebook 05 fresh-rerun QC completed."
)

print(
    "Article-selected resolution:",
    ARTICLE_SELECTED_RESOLUTION,
)

print(
    "Archived article r=0.60 cluster count:",
    HISTORICAL_R06_N_CLUSTERS,
)

print(
    "Fresh-rerun r=0.60 cluster count:",
    fresh_r06_n_clusters,
)

print(
    "Fresh-rerun exact archived structural match:",
    fresh_rerun_reproduces_archived_size_summary,
)

print(
    "Article downstream source:",
    status_df.iloc[0][
        "article_downstream_source"
    ],
)

[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\data\processed\integrated\GBM_core_4datasets_HVG8000_scVI_clustering_resolution_QC_rerun_lowmem.h5ad
[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\clustering_resolution_QC_output_manifest_HVG8000_rerun_lowmem.tsv shape=(15, 3)


,name,path,exists
0,input_scvi_h5ad,G:\Repository\glioma-cnv-single-cell-atlas\dat...,True
1,input_hvg_counts_h5ad,G:\Repository\glioma-cnv-single-cell-atlas\dat...,True
2,fullgene_h5ad,G:\Repository\glioma-cnv-single-cell-atlas\dat...,True
3,scratch_checkpoint_h5ad,E:\glioma-cnv-scratch\main_analysis_05\05_leid...,True
4,fresh_rerun_resolution_qc_h5ad,G:\Repository\glioma-cnv-single-cell-atlas\dat...,True
5,input_qc_tsv,G:\Repository\glioma-cnv-single-cell-atlas\met...,True
6,sweep_summary_tsv,G:\Repository\glioma-cnv-single-cell-atlas\met...,True
7,cluster_sizes_tsv,G:\Repository\glioma-cnv-single-cell-atlas\met...,True
8,composition_tsv,G:\Repository\glioma-cnv-single-cell-atlas\met...,True
9,composition_summary_tsv,G:\Repository\glioma-cnv-single-cell-atlas\met...,True



DONE — notebook 05 fresh-rerun QC completed.
Article-selected resolution: 0.6
Archived article r=0.60 cluster count: 18
Fresh-rerun r=0.60 cluster count: 15
Fresh-rerun exact archived structural match: False
Article downstream source: canonical_historical_artifact_required


## Expected completion

The final cell reports both the archived article checkpoint and the fresh-rerun
result. A fresh stochastic rerun is **not** required to reproduce the archived
partition exactly.

For the August 2026 audit on the current Windows/GPU rerun, `r=0.60` produced
15 clusters, while the archived executed article-defining run contained 18.
Separate read-only audits established that the runs used the same 351,427
biological cells, the same ordered 8,000 HVGs, semantically identical raw count
vectors for every cell, identical categorical scVI covariates after nomenclature
normalization, and numerically equivalent continuous covariates.

Therefore this notebook writes only **fresh-rerun** QC artifacts. When the fresh
rerun does not reproduce the archived structural checkpoint, the exact article
branch must use the canonical historical scVI/Leiden artifact.